# Projeto: Inserção dos dados na camada Gold do banco de dados.

### Descrição Geral
Este notebook executa o processo de ingestão e enriquecimento dos dados climáticos, a partir das saídas validadas pelas regras do Great Expectations (GX).<br>
As tabelas da camada silver são previamente avaliadas por um pipeline de qualidade. As linhas que passam pelas validações são salvas em diretórios separados (arquivos de válidos), que são então lidos neste notebook para composição da camada gold.<br>
Após o carregamento dos dados qualificados, realiza-se o tratamento padronizado e o enriquecimento dos campos, com integração entre múltiplas fontes. O resultado final é persistido na tabela `grupo01.gold.clima`, já consolidada e pronta para análises.

### Informações Técnicas

- **Fonte de Dados**:
  * Tabelas das bases Silver referentes a previsões climática
    * `clima_tempo`
    * `inpe`
    * `open_weather_map`
    * `weather`
    * `weather_api`
- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks

- **Dados persistidos**:
  - Data de carga
  - Data da previsão
  - Nome do município e UF
  - Temperatura máxima e mínima
  - Umidade máxima e mínima
  - Precipitação
  - Propabilidade de precipitação
  - Velocidade do vento
  - Fonte de previsão
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 18/05/2025 | Lucas Berta e Lucas Bringmann     |  1.0   | Criação da inserção dos dados para camada gold|
| 20/05/2025 | Gabriel Rost      |  1.1   | Inclusão de documentação sobre enrequecimentos realizados|
| 25/05/2025 | Evelise Dias      |  1.2  | 	Adição da leitura de arquivos validados pelo Great Expectations|
| 26/05/2025 | Evelise Dias      |  1.3  | 	Atualização da descrição geral do projeto e alinhamento com a arquitetura atual|

In [0]:
%run ../../src/utils/utils_run_gold

In [0]:
%run ./queries_gold

In [0]:
pasta_validos_clima_tempo = lakehouse_logs_gx + "/silver/clima_tempo/validos/clima_tempo"
df_clima_tempo = ler_tabelas(pasta_validos_clima_tempo, fromTable=False)

In [0]:
pasta_validos_weather = lakehouse_logs_gx + "/silver/weather/validos/weather"
df_weather = ler_tabelas(pasta_validos_weather, fromTable=False)

In [0]:
pasta_validos_weather_api = lakehouse_logs_gx + "/silver/weather_api/validos/weather_api"
df_weather_api = ler_tabelas(pasta_validos_weather_api, fromTable=False)

In [0]:
pasta_validos_open_weather_map_api = lakehouse_logs_gx + "/silver/open_weather_map/validos/open_weather_map"
df_open_weather_map_api = ler_tabelas(pasta_validos_open_weather_map_api, fromTable=False)

In [0]:
pasta_validos_inpe = lakehouse_logs_gx + "/silver/inpe/validos/inpe"
df_inpe = ler_tabelas(pasta_validos_inpe, fromTable=False)

In [0]:
df_inpe_padronizado = (
    df_inpe
    .withColumn("umid_max", lit(None).cast("decimal(5,2)"))
    .withColumn("umid_min", lit(None).cast("decimal(5,2)"))
    .withColumn("vel_vento", lit(None).cast("decimal(5,2)"))
)

In [0]:
df_open_weather_map_api = (
    df_open_weather_map_api
    .withColumn("prob_precipitacao", lit(None).cast("decimal(5,2)"))
)

In [0]:
df_final = df_clima_tempo.unionByName(df_inpe_padronizado)

In [0]:
df_final = df_final.unionByName(df_open_weather_map_api)

In [0]:
df_final = df_final.unionByName(df_weather_api)

In [0]:
df_final = df_final.unionByName(df_weather)

In [0]:
spark.sql(create_table_gold_clima)

In [0]:
df_sem_duplicatas = remover_duplicatas(df_final, ["id", "dt_previsao", "nom_municipio", "flg_fonte_previsao"])

In [0]:
mapping_expr = create_map([lit(x) for pair in MAPPING_ESTADOS.items() for x in pair])

In [0]:
df_updated = df_sem_duplicatas.withColumn("sig_uf", mapping_expr[col("sig_uf")])

In [0]:
df_updated = df_updated.withColumnRenamed("sig_uf", "nom_estado")

In [0]:
inserir_com_merge(dataframe = df_updated, tabela_destino = "grupo01.gold.clima", camposChave=["id"])